In [4]:
# %% [markdown]
# ================================================================================
# MULTIMODAL LIFE EXPECTANCY PREDICTION — PUBLICATION FIGURES (v5)
# Unified plotting script for reviewer-requested diagnostics
# ================================================================================

# %%
# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS & GLOBAL STYLE
# ─────────────────────────────────────────────────────────────────────────────
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from pathlib import Path
from scipy import stats

warnings.filterwarnings("ignore")

# ── Data & Output directories (Mapped to your Mac environment) ───────────────
DATA_DIR = Path('/Users/faizahmad/Desktop/reviewer_ml_re')
OUT      = Path('/Users/faizahmad/Desktop/paper1 2026-04-18 /paper1_run1/figures')
OUT.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 1. GLOBAL STYLE CONSTANTS (Fonts gently increased)
# ─────────────────────────────────────────────────────────────────────────────
FONT_FAMILY   = ["Helvetica Neue", "Arial", "DejaVu Sans"]
FS_BASE       = 15   # Increased from 14
FS_LABEL      = 18
FS_TITLE      = 16
FS_SUPTITLE   = 18
FS_PANEL_TAG  = 16
FS_ANNOT      = 13   # Increased from 11
FS_LEGEND     = 13   # Increased from 11

DPI_SCREEN    = 150
DPI_SAVE      = 600  # High DPI for publication-quality output
PAD           = 0.18 

C = dict(
    blue    = "#0072B2",
    orange  = "#E69F00",
    green   = "#009E73",
    red     = "#D55E00",
    purple  = "#CC79A7",
    sky     = "#56B4E9",
    yellow  = "#F0E442",
    black   = "#111111",
    grey    = "#AAAAAA",
    night   = "#1A1A4E",
    day     = "#FF8C00",
    lgrey   = "#EEEEEE",
)

plt.rcParams.update({
    "font.family":           "sans-serif",
    "font.sans-serif":       FONT_FAMILY,
    "font.size":             FS_SUPTITLE,
    "axes.titlesize":        FS_TITLE,
    "axes.labelsize":        FS_LABEL,
    "xtick.labelsize":       FS_BASE,
    "ytick.labelsize":       FS_BASE,
    "legend.fontsize":       FS_LEGEND,
    "figure.dpi":            DPI_SCREEN,
    "savefig.dpi":           DPI_SAVE,
    "savefig.bbox":          "tight",
    "savefig.pad_inches":    PAD,
    "pdf.fonttype":          42,
    "ps.fonttype":           42,
    "axes.linewidth":        1.2,
    "grid.alpha":            0.4,
    "grid.linewidth":        1.5,
    "grid.linestyle":        "--",
    "axes.titleweight":      "bold",
    "image.cmap":            "viridis",
    "legend.frameon":        True,
    "axes.spines.top":       False,
    "axes.spines.right":     False,
})
plt.style.use("seaborn-v0_8-white")

# ─────────────────────────────────────────────────────────────────────────────
# 2. SHARED UTILITY FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────
def save_fig(fig, name):
    p = OUT / name
    fig.savefig(p.with_suffix(".pdf"))
    fig.savefig(p.with_suffix(".png"))
    plt.close(fig)
    print(f"  ✓ Saved: {name}.pdf / .png")

def lowess_xy(x, y, frac=0.3):
    from statsmodels.nonparametric.smoothers_lowess import lowess
    m = np.isfinite(x) & np.isfinite(y)
    o = np.argsort(x[m])
    lw = lowess(y[m][o], x[m][o], frac=frac, return_sorted=True)
    return lw[:, 0], lw[:, 1]

def resolve(fn_list, *candidates):
    low = {f.lower(): f for f in fn_list}
    for c in candidates:
        if c in fn_list: return c
        if c.lower() in low: return low[c.lower()]
    for c in candidates:
        for f in fn_list:
            if c.lower().replace(" ", "") in f.lower().replace(" ", ""):
                return f
    return None

def reconstruct_sorted_df(data_file):
    df = pd.read_csv(data_file)
    if "MeanLifeExpectency_x" in df.columns:
        df["MeanLifeExpectency"] = df["MeanLifeExpectency_x"]
    df = df.dropna(subset=["MeanLifeExpectency"]).reset_index(drop=True)
    df["fips"] = df["fips"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
    df = df[~df["fips"].str.startswith(("02", "15"))].reset_index(drop=True)
    df = df.sort_values(["fips", "year"]).reset_index(drop=True)
    return df

# %%
# ─────────────────────────────────────────────────────────────────────────────
# 3. LOAD CORE DATA
# ─────────────────────────────────────────────────────────────────────────────
print("Loading core data files. This may take a moment for the 4.3GB PKL...")

RESULTS_FILE = DATA_DIR / "ml_results.pkl"
with open(RESULTS_FILE, "rb") as f:
    R = pickle.load(f)

shap_vals   = R["shap_values"]            
shap_sample = R["shap_sample"]            
fn_list     = list(shap_sample.columns)
predictions = R["predictions"].copy()

df_sorted = reconstruct_sorted_df(DATA_DIR / "full_clean_engineered_dataset_with_LE.csv")
sample_pos = shap_sample.index.values

print("Data loaded successfully.")

# %%
# ─────────────────────────────────────────────────────────────────────────────
# 4. PLOTTING FUNCTIONS 
# ─────────────────────────────────────────────────────────────────────────────

def plot_generalisation_stress_test():
    print("Plotting Generalisation Stress Test...")
    res_file = DATA_DIR / "reviewer_diagnostics_outputs" / "reviewer_diagnostics_master_results.csv"
    if not res_file.exists(): return print("File not found.")
    res = pd.read_csv(res_file)
    
    wanted = [("A_naive", "Naive mean"), ("A_spatial_knn", "Spatial k-NN (k=4, IDW)"),
              ("C_state_groupcv", "Leave-states-out CV"), ("C_temporal_holdout", "Temporal holdout (2015-19)"),
              ("D_temporally_complete", "Always-available sources"), ("D_modern_only", "Modern only (2015-19)"),
              ("Production", "Production (county-grouped)")]
    rows = []
    for analysis, label in wanted:
        sub = res[res["Analysis"] == analysis]
        if analysis == "A_spatial_knn": sub = sub[sub["Variant"].str.startswith("k=4_idw")]
        if analysis == "C_temporal_holdout": sub = sub[sub["Variant"].str.contains("2015")]
        if len(sub):
            r = sub.iloc[0]
            rows.append((label, r["R2_mean"], r.get("R2_sd", 0), r["MAE_mean"], r.get("MAE_sd", 0)))
            
    labels = [r[0] for r in rows]; r2 = [r[1] for r in rows]; r2e = [r[2] for r in rows]
    mae = [r[3] for r in rows]; maee = [r[4] for r in rows]
    colors = [C["grey"] if "Naive" in l or "k-NN" in l else C["blue"] for l in labels]

    fig, ax = plt.subplots(1, 2, figsize=(15, 6.5))
    x = np.arange(len(labels))
    ax[0].bar(x, r2, yerr=r2e, capsize=4, color=colors, edgecolor="white")
    ax[0].axhline(0.82, ls="--", color=C["red"], lw=1.6, label="IHME socio-demographic R$^2$=0.82")
    ax[0].set_ylabel("R$^2$ (test)", fontweight="bold")
    ax[0].set_title("(A) R$^2$ across baselines & validation regimes", fontweight="bold", loc="left")
    ax[0].legend(fontsize=FS_LEGEND)
    
    ax[1].bar(x, mae, yerr=maee, capsize=4, color=colors, edgecolor="white")
    ax[1].set_ylabel("MAE (years)", fontweight="bold")
    ax[1].set_title("(B) Mean absolute error", fontweight="bold", loc="left")
    
    for a in ax:
        a.set_xticks(x)
        # Rotated to 45 degrees to prevent overlap, keeping right-aligned
        a.set_xticklabels(labels, rotation=45, ha="right", fontsize=FS_ANNOT)
        a.grid(axis="y", alpha=0.25)
        
    fig.suptitle("Generalisation stress-test: baselines vs the satellite model", fontweight="bold", y=1.05)
    plt.tight_layout()
    save_fig(fig, "figR_generalisation_stress_test")


def plot_nce_decomposition():
    print("Plotting NCE Decomposition...")
    nce = resolve(fn_list, "Nighttime Cooling Efficiency", "ENG_Night_Cooling_Eff", "NCE")
    j = fn_list.index(nce)
    nce_shap = shap_vals[:, j]
    nce_val = shap_sample[nce].values
    day_p90 = df_sorted["LST_Day_1km_p90"].iloc[sample_pos].values
    night_p10 = df_sorted["LST_Night_1km_p10"].iloc[sample_pos].values

    fig, ax = plt.subplots(1, 3, figsize=(21, 6.2))
    lx, ly = lowess_xy(nce_val, nce_shap)

    sc0 = ax[0].scatter(nce_val, nce_shap, c=night_p10, cmap="coolwarm", s=14, alpha=0.5, rasterized=True)
    ax[0].plot(lx, ly, color=C["night"], lw=3, zorder=5)
    ax[0].axhline(0, color=C["grey"], lw=1, ls="--")
    cb0 = fig.colorbar(sc0, ax=ax[0], fraction=0.046, pad=0.02)
    cb0.set_label("Nighttime LST P10 (°C)", fontsize=FS_ANNOT)
    ax[0].set_xlabel("NCE value (unitless)", fontweight="bold")
    ax[0].set_ylabel("SHAP value (Δ LE, years)", fontweight="bold")
    ax[0].set_title("(A) NCE dependence, coloured by cold-night component", fontweight="bold", loc="left")

    sc1 = ax[1].scatter(day_p90, night_p10, c=nce_val, cmap="viridis", s=14, alpha=0.6, rasterized=True)
    cb1 = fig.colorbar(sc1, ax=ax[1], fraction=0.046, pad=0.02)
    cb1.set_label("NCE value", fontsize=FS_ANNOT)
    ax[1].set_xlabel("Daytime LST P90 (°C)", fontweight="bold")
    ax[1].set_ylabel("Nighttime LST P10 (°C)", fontweight="bold")
    ax[1].set_title("(B) Physical meaning of NCE\n(high NCE = hot days + cold nights)", fontweight="bold", loc="left")

    sc2 = ax[2].scatter(nce_val, nce_shap, c=day_p90, cmap="YlOrRd", s=14, alpha=0.5, rasterized=True)
    ax[2].plot(lx, ly, color=C["red"], lw=3, zorder=5)
    ax[2].axhline(0, color=C["grey"], lw=1, ls="--")
    cb2 = fig.colorbar(sc2, ax=ax[2], fraction=0.046, pad=0.02)
    cb2.set_label("Daytime LST P90 (°C)", fontsize=FS_ANNOT)
    ax[2].set_xlabel("NCE value (unitless)", fontweight="bold")
    ax[2].set_ylabel("SHAP value (Δ LE, years)", fontweight="bold")
    ax[2].set_title("(C) NCE dependence, coloured by hot-day component", fontweight="bold", loc="left")

    for a in ax: a.grid(True, alpha=0.2)
    fig.suptitle("Nighttime Cooling Efficiency (NCE) = (P90$_{day}$ − P10$_{night}$) / (P90$_{day}$ + ε)", fontweight="bold", y=1.03)
    plt.tight_layout()
    save_fig(fig, "figR_NCE_decomposition")


def plot_orbital_drift():
    print("Plotting Orbital Drift...")
    key = "LST_Night_1km_p10"
    nat = df_sorted.groupby("year")["LST_Night_1km_mean"].mean()
    years = nat.index.values.astype(float)
    vals = nat.values
    sl, ic, r, p_lin, se = stats.linregress(years, vals)

    g = df_sorted.groupby("year")[key]
    between = np.var(g.mean().values, ddof=0)
    within = g.var(ddof=0).mean()
    total = df_sorted[key].var(ddof=0)

    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    ax[0].plot(years, vals, "o-", color=C["night"], lw=2.5, ms=7)
    ax[0].plot(years, sl * years + ic, "--", color=C["red"], lw=1.8)
    ax[0].set_xlabel("Year", fontweight="bold")
    ax[0].set_ylabel("National mean nighttime LST (°C)", fontweight="bold")
    ax[0].set_title("(A) No material national LST trend over 20 years", fontweight="bold", loc="left")
    ax[0].text(0.04, 0.06, f"OLS slope = {sl:+.4f} °C/yr (p = {p_lin:.2f})", transform=ax[0].transAxes, fontsize=FS_ANNOT, va="bottom", bbox=dict(boxstyle="round,pad=0.4", fc="white", ec=C["grey"]))
    ax[0].grid(True, alpha=0.25)

    parts = ["Cross-sectional\n(within-year,\ncounty-to-county)", "Temporal\n(between-year,\nnational means)"]
    vals_bar = [within, between]
    ax[1].bar([0, 1], vals_bar, color=[C["blue"], C["orange"]], edgecolor="white", width=0.6)
    for i, v in enumerate(vals_bar):
        ax[1].text(i, v, f"{v:.2f}\n({100*v/total:.1f}% of total)", ha="center", va="bottom", fontsize=FS_ANNOT, fontweight="bold")
    ax[1].set_xticks([0, 1])
    ax[1].set_xticklabels(parts, fontsize=FS_ANNOT)
    ax[1].set_ylabel(f"Variance of {key}", fontweight="bold")
    ax[1].set_title("(B) The predictive signal is spatial, not temporal", fontweight="bold", loc="left")
    ax[1].set_ylim(0, max(vals_bar) * 1.25)
    ax[1].grid(axis="y", alpha=0.25)

    fig.suptitle("MODIS Terra orbital drift vs. County-to-County contrast", fontweight="bold", y=1.02)
    plt.tight_layout()
    save_fig(fig, "figR_orbital_drift")


def plot_reservation_audit():
    print("Plotting Reservation Audit...")
    res_file = DATA_DIR / "reviewer_diagnostics_outputs" / "reservation_fold_audit.csv"
    if not res_file.exists(): return print("File not found.")
    audit = pd.read_csv(res_file)
    nat_resid = predictions["residual"].mean()

    fig, ax = plt.subplots(figsize=(11, 6.5))
    order = audit.sort_values("Mean_residual")
    yy = np.arange(len(order))
    cols = [C["red"] if v < 0 else C["blue"] for v in order["Mean_residual"]]
    
    ax.barh(yy, order["Mean_residual"], color=cols, edgecolor="white")
    ax.axvline(0, color=C["black"], lw=1.2)
    ax.axvline(nat_resid, color=C["green"], ls="--", lw=1.6, label=f"National mean residual = {nat_resid:+.2f} yr")
    ax.set_yticks(yy)
    # Brackets safely stripped from the fold output here
    ax.set_yticklabels([f"{r.County} ({r.FIPS}, fold {str(r.Tested_in_fold).strip('[]')})" for r in order.itertuples()], fontsize=FS_ANNOT)
    ax.set_xlabel("Mean residual (Actual − Predicted, years)", fontweight="bold")
    ax.set_title("Indigenous-reservation counties (The 'Spectral Shadow')", fontweight="bold", loc="left")
    ax.legend(fontsize=FS_LEGEND)
    ax.grid(axis="x", alpha=0.25)
    
    plt.tight_layout()
    save_fig(fig, "figR_reservation_audit")

def plot_forest_attenuation():
    print("Plotting Forest Attenuation...")
    lst = resolve(fn_list, "Daytime Surface Temp (Mean)", "LST_Day_1km_mean", "Daytime LST Mean")
    forest = resolve(fn_list, "Deciduous Forest %", "USDA_Cropland_USDA_Forest_Deciduous_pct", "Deciduous Forest")
    li = fn_list.index(lst)
    lst_x = shap_sample[lst].values
    lst_s = shap_vals[:, li]
    fv = shap_sample[forest].values

    labs = ["Low (<5%)", "Medium (5–20%)", "High (>20%)"]
    bins = [-np.inf, 5, 20, np.inf] if np.nanmax(fv) > 1.5 else [-np.inf, 0.05, 0.20, np.inf]
    fcat = pd.cut(fv, bins=bins, labels=labs)
    q = pd.qcut(lst_x, 4, labels=["Q1", "Q2", "Q3", "Q4"])
    d = pd.DataFrame({"q": q, "f": fcat, "s": lst_s}).dropna()

    s_low = d[(d.f == "Low (<5%)") & (d.q == "Q4")]["s"].median()
    s_high = d[(d.f == "High (>20%)") & (d.q == "Q4")]["s"].median()
    
    # CALCULATE ABSOLUTE DIFFERENCE ONLY (Fixing the 125% / 0% logic)
    delta = abs(s_high - s_low)

    pal = {"Low (<5%)": C["red"], "Medium (5–20%)": C["orange"], "High (>20%)": C["green"]}
    fig, ax = plt.subplots(1, 2, figsize=(16, 6.5))
    
    for cat in labs:
        m = fcat == cat
        if m.sum() > 50:
            lx, ly = lowess_xy(lst_x[m], lst_s[m])
            ax[0].plot(lx, ly, color=pal[cat], lw=3, label=cat)
            
    ax[0].axhline(0, color=C["grey"], lw=1.5)
    ax[0].set_xlabel("Daytime LST (°C)", fontweight="bold")
    ax[0].set_ylabel("Daytime-LST SHAP (years)", fontweight="bold")
    ax[0].set_title("(A) Heat penalty by forest stratum", fontweight="bold", loc="left")
    ax[0].legend(title="Deciduous forest cover", fontsize=FS_LEGEND)
    ax[0].grid(True, alpha=0.25)

    pos = np.arange(4)
    offs = {"Low (<5%)": -0.25, "Medium (5–20%)": 0, "High (>20%)": 0.25}
    for cat in labs:
        meds = [d[(d.f == cat) & (d.q == f"Q{i+1}")]["s"].median() for i in range(4)]
        ax[1].bar(pos + offs[cat], meds, width=0.22, color=pal[cat], edgecolor="white", label=cat)
        
    ax[1].axhline(0, color=C["grey"], lw=1.5)
    ax[1].set_xticks(pos)
    ax[1].set_xticklabels(["Q1 (cool)", "Q2", "Q3", "Q4 (hot)"], fontsize=FS_ANNOT)
    ax[1].set_xlabel("Daytime LST quartile", fontweight="bold")
    ax[1].set_ylabel("Median daytime-LST SHAP (years)", fontweight="bold")
    
    # Update the text box to ONLY show the absolute difference
    txt = f"Q4 absolute ΔSHAP = {delta:.3f} yr"
    ax[1].text(0.97, 0.05, txt, transform=ax[1].transAxes, ha="right", va="bottom", 
               fontsize=FS_ANNOT, fontweight="bold", 
               bbox=dict(boxstyle="round,pad=0.5", fc="#E8F5E9", ec=C["green"]))
               
    ax[1].set_title("(B) Absolute SHAP difference in the hottest quartile", fontweight="bold", loc="left")
    ax[1].legend(title="Forest cover", fontsize=FS_LEGEND, loc="upper center", ncol=3)
    ax[1].grid(axis="y", alpha=0.25)

    fig.suptitle("Forest cover interactions with daytime-heat SHAP penalty", fontweight="bold", y=1.02)
    plt.tight_layout()
    save_fig(fig, "figR_forest_attenuation_fixed")


def plot_nightlights_ablation():
    print("Plotting Nightlights Ablation...")
    res_file = DATA_DIR / "nightlights_pm25_outputs" / "nightlights_pm25_ablation_results.csv"
    if not res_file.exists(): return print("File not found.")
    res = pd.read_csv(res_file)
    
    sub = res[res["Analysis"] == "NTL_PM25"].copy()
    if sub.empty: return
    order = sub.sort_values("R2_mean")
    
    # Reviewer tags cleanly removed from labels
    labels = {
        "full_reference": "Production (satellite-only)",
        "full_plus_NTL_PM25": "Production + NTL + PM2.5",
        "full_plus_NTL": "Production + NTL",
        "full_plus_PM25": "Production + PM2.5",
        "NTL_plus_PM25_only": "NTL + PM2.5 only",
        "NTL_only": "NTL only",
        "PM25_only": "PM2.5 only",
    }
    names = [labels.get(v, v) for v in order["Variant"]]
    
    fig, ax = plt.subplots(figsize=(10, 0.7 * len(order) + 2))
    colors = [C["blue"] if "Production" in n and "only" not in n else C["orange"] for n in names]
    ax.barh(range(len(order)), order["R2_mean"], xerr=order["R2_sd"], color=colors, alpha=0.9, edgecolor="white", capsize=4)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(names, fontsize=FS_ANNOT)
    ax.set_xlabel("Cross-validated $R^2$ (county-grouped 5-fold)", fontweight="bold")
    ax.set_title("Nighttime lights & PM2.5: feature-block ablation", fontweight="bold")
    
    for i, (r2, sd) in enumerate(zip(order["R2_mean"], order["R2_sd"])):
        ax.text(r2 + sd + 0.005, i, f"{r2:.3f}", va="center", fontsize=FS_ANNOT)
        
    fig.tight_layout()
    save_fig(fig, "figR_nightlights_pm25_ablation")


def plot_learner_comparison():
    print("Plotting Learner Comparison...")
    res_file = DATA_DIR / "multilearner_parallel_outputs" / "multilearner_results.csv"
    if not res_file.exists(): return print("File not found.")
    res = pd.read_csv(res_file)
    
    order = res.sort_values("R2_mean")
    fig, ax = plt.subplots(figsize=(9, 0.7 * len(order) + 2))
    ax.barh(range(len(order)), order["R2_mean"], xerr=order["R2_sd"], color=C["blue"], alpha=0.9, edgecolor="white", capsize=4)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order["Learner"], fontsize=FS_ANNOT)
    ax.set_xlabel("Cross-validated $R^2$ (county-grouped 5-fold)", fontweight="bold")
    
    # Reviewer tag securely removed from title
    ax.set_title("Learner comparison\nAll learners share identical features and CV", fontweight="bold")
    
    for i, (r2, sd) in enumerate(zip(order["R2_mean"], order["R2_sd"])):
        ax.text(r2 + sd + 0.004, i, f"{r2:.3f}", va="center", fontsize=FS_ANNOT)
        
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    save_fig(fig, "figR_learner_comparison")

# %%
# ─────────────────────────────────────────────────────────────────────────────
# 5. EXECUTE ALL PLOTS
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    plot_generalisation_stress_test()
    plot_nce_decomposition()
    plot_orbital_drift()
    plot_reservation_audit()
    plot_forest_attenuation()
    plot_nightlights_ablation()
    plot_learner_comparison()
    print("\n✅ All visual styling unified and plots successfully saved!")

Loading core data files. This may take a moment for the 4.3GB PKL...
Data loaded successfully.
Plotting Generalisation Stress Test...
  ✓ Saved: figR_generalisation_stress_test.pdf / .png
Plotting NCE Decomposition...
  ✓ Saved: figR_NCE_decomposition.pdf / .png
Plotting Orbital Drift...
  ✓ Saved: figR_orbital_drift.pdf / .png
Plotting Reservation Audit...
  ✓ Saved: figR_reservation_audit.pdf / .png
Plotting Forest Attenuation...
  ✓ Saved: figR_forest_attenuation_fixed.pdf / .png
Plotting Nightlights Ablation...
  ✓ Saved: figR_nightlights_pm25_ablation.pdf / .png
Plotting Learner Comparison...
  ✓ Saved: figR_learner_comparison.pdf / .png

✅ All visual styling unified and plots successfully saved!
